In [14]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS silver
""")

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 19, Finished, Available, Finished, False)

DataFrame[]

### cleaned & transformed customer table

In [15]:
silver_customer = spark.table("bronze.brz_customer")
silver_customer.show(10)

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 20, Finished, Available, Finished, False)

+-----------+--------------------+-------------+---------------+-------------+---------+--------------------+--------------------+
|CUSTOMER_ID|       CUSTOMER_NAME|REGISTERED_ON|KYC_VERIFIED_ON|CUSTOMER_TYPE|     CITY|         ingested_at|         source_file|
+-----------+--------------------+-------------+---------------+-------------+---------+--------------------+--------------------+
|     C20000|          Diya Mehta|   2024/10/09|     2024-11-22|   INDIVIDUAL| Lonavala|2026-10-13 10:47:...|abfss://c9364e95-...|
|     C20001|      Sterling Infra|   21-06-2023|     2023-07-06|      COMPANY| Baramati|2026-10-13 10:47:...|abfss://c9364e95-...|
|     C20002| Trident Civil Works|   2023/06/14|     2023-07-13|      COMPANY|Chinchwad|2026-10-13 10:47:...|abfss://c9364e95-...|
|     C20003|        Rahul Mehta |   10-10-2024|     2024-10-16|   contractor|  Kharadi|2026-10-13 10:47:...|abfss://c9364e95-...|
|     C20004|       Vivaan Pillai|   14-01-2025|     2025-02-19|   contractor| Lona

In [16]:
silver_customer.groupBy('customer_id').count().filter(F.col('count')>=2).show()

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 21, Finished, Available, Finished, False)

+-----------+-----+
|customer_id|count|
+-----------+-----+
|     C20277|    2|
|     C20054|    2|
+-----------+-----+



In [17]:
silver_customer = silver_customer.dropDuplicates(subset=['customer_id'])

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 22, Finished, Available, Finished, False)

In [18]:
silver_customer.groupBy('customer_id').count().filter(F.col('count')>=2).show()

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 23, Finished, Available, Finished, False)

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [19]:
silver_customer.select('REGISTERED_ON').distinct().show()
silver_customer.select('KYC_VERIFIED_ON').distinct().show()
silver_customer.select('CUSTOMER_TYPE').distinct().show()

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 24, Finished, Available, Finished, False)

+-------------+
|REGISTERED_ON|
+-------------+
|   2025/11/15|
|   2022/12/26|
|   2020/07/28|
|   2023/08/27|
|   2022/10/25|
|   2025/02/03|
|   2021/02/21|
|   2021/06/09|
|   07-09-2022|
|   2022/09/22|
|   08-07-2024|
|   2020/04/23|
|   04-04-2023|
|   2020/08/11|
|   09-10-2020|
|   12-09-2024|
|   2024/08/21|
|   01-07-2020|
|   12-01-2022|
|   16-12-2023|
+-------------+
only showing top 20 rows
+---------------+
|KYC_VERIFIED_ON|
+---------------+
|     2024-10-24|
|     2023-05-01|
|     2024-09-15|
|     2020-06-24|
|     2023-04-28|
|     2023-04-21|
|     2021-12-23|
|     2024-12-12|
|     2025-03-20|
|     2025-09-14|
|     2025-04-05|
|     2024-02-08|
|     2024-02-28|
|     2023-03-10|
|     2023-10-26|
|     2025-01-03|
|     2023-05-04|
|     2025-08-18|
|     2025-06-16|
|     2023-10-18|
+---------------+
only showing top 20 rows
+-------------+
|CUSTOMER_TYPE|
+-------------+
|       Contr.|
|   Contractor|
|      Company|
|   Individual|
|   contractor|
|     

In [20]:
# correct 'registered_on'  date in the unified format

silver_customer = silver_customer.withColumn(
    'REGISTERED_ON',
    F.coalesce(
        F.to_date(F.col('REGISTERED_ON'),'yyyy/MM/dd'),
        F.to_date(F.col('REGISTERED_ON'),'dd-MM-yyyy'),
        F.to_date(F.col('REGISTERED_ON'),'yyyy-MM-dd')
    )    
)

silver_customer.show(10)



StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 25, Finished, Available, Finished, False)

+-----------+--------------------+-------------+---------------+-------------+---------+--------------------+--------------------+
|CUSTOMER_ID|       CUSTOMER_NAME|REGISTERED_ON|KYC_VERIFIED_ON|CUSTOMER_TYPE|     CITY|         ingested_at|         source_file|
+-----------+--------------------+-------------+---------------+-------------+---------+--------------------+--------------------+
|     C20000|          Diya Mehta|   2024-10-09|     2024-11-22|   INDIVIDUAL| Lonavala|2026-10-13 10:47:...|abfss://c9364e95-...|
|     C20001|      Sterling Infra|   2023-06-21|     2023-07-06|      COMPANY| Baramati|2026-10-13 10:47:...|abfss://c9364e95-...|
|     C20002| Trident Civil Works|   2023-06-14|     2023-07-13|      COMPANY|Chinchwad|2026-10-13 10:47:...|abfss://c9364e95-...|
|     C20003|        Rahul Mehta |   2024-10-10|     2024-10-16|   contractor|  Kharadi|2026-10-13 10:47:...|abfss://c9364e95-...|
|     C20004|       Vivaan Pillai|   2025-01-14|     2025-02-19|   contractor| Lona

In [21]:
# uniform the 'customer_type'abs

silver_customer.select('customer_type').distinct().show()

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 26, Finished, Available, Finished, False)

+-------------+
|customer_type|
+-------------+
|       Contr.|
|   Contractor|
|      Company|
|   Individual|
|   contractor|
|      COMPANY|
|    Corporate|
|   CONTRACTOR|
|      company|
|          IND|
|         CORP|
|   individual|
|   INDIVIDUAL|
+-------------+



In [22]:
correct_customer_type = {
    'Contr.':'Contractor',
    'IND':'Individual',
    'CORP':'Corporate'
}

silver_customer = silver_customer.replace(correct_customer_type,subset=['CUSTOMER_TYPE'])

silver_customer = silver_customer.withColumn(
    'CUSTOMER_TYPE',
    F.initcap('CUSTOMER_TYPE')
)

silver_customer = silver_customer.withColumn(
    'KYC_VERIFIED_ON',
    F.to_date(F.col('KYC_VERIFIED_ON'),'yyyy-MM-dd')
)

silver_customer.show()

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 27, Finished, Available, Finished, False)

+-----------+--------------------+-------------+---------------+-------------+---------+--------------------+--------------------+
|CUSTOMER_ID|       CUSTOMER_NAME|REGISTERED_ON|KYC_VERIFIED_ON|CUSTOMER_TYPE|     CITY|         ingested_at|         source_file|
+-----------+--------------------+-------------+---------------+-------------+---------+--------------------+--------------------+
|     C20000|          Diya Mehta|   2024-10-09|     2024-11-22|   Individual| Lonavala|2026-10-13 10:47:...|abfss://c9364e95-...|
|     C20001|      Sterling Infra|   2023-06-21|     2023-07-06|      Company| Baramati|2026-10-13 10:47:...|abfss://c9364e95-...|
|     C20002| Trident Civil Works|   2023-06-14|     2023-07-13|      Company|Chinchwad|2026-10-13 10:47:...|abfss://c9364e95-...|
|     C20003|        Rahul Mehta |   2024-10-10|     2024-10-16|   Contractor|  Kharadi|2026-10-13 10:47:...|abfss://c9364e95-...|
|     C20004|       Vivaan Pillai|   2025-01-14|     2025-02-19|   Contractor| Lona

In [23]:
# uniform the 'customer_type'abs - double check

silver_customer.select('customer_type').distinct().show()

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 29, Finished, Available, Finished, False)

+-------------+
|customer_type|
+-------------+
|   Contractor|
|      Company|
|   Individual|
|    Corporate|
+-------------+



In [24]:
silver_customer.printSchema()

print(f'{silver_customer.count()}','<< silver_customer_count')

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 31, Finished, Available, Finished, False)

root
 |-- CUSTOMER_ID: string (nullable = true)
 |-- CUSTOMER_NAME: string (nullable = true)
 |-- REGISTERED_ON: date (nullable = true)
 |-- KYC_VERIFIED_ON: date (nullable = true)
 |-- CUSTOMER_TYPE: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

600 << silver_customer_count


In [25]:
# save that into silver layer

silver_customer.write\
.format('delta')\
.mode('overwrite')\
.option('mergeSchema','true')\
.saveAsTable('silver.slv_customer')


StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 34, Finished, Available, Finished, False)

### Cleaned & Transformed billing table


In [26]:
silver_billing = spark.table("bronze.brz_billing")
silver_billing.show(10)

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 35, Finished, Available, Finished, False)

+-------+---------+----------+--------------+----------+--------------------+--------------------+
|BILL_ID|RENTAL_ID| BILL_DATE|    AMOUNT_INR|PAYER_TYPE|         ingested_at|         source_file|
+-------+---------+----------+--------------+----------+--------------------+--------------------+
| B50612|  R700624|2026-06-23| Rs. 64,157.16|    DIRECT|2026-10-13 10:47:...|abfss://c9364e95-...|
| B50483|  R700487|2026-06-20|     26,037.89| CORPORATE|2026-10-13 10:47:...|abfss://c9364e95-...|
| B50362|  R700365|2026-06-22|Rs. 174,898.87|  Contract|2026-10-13 10:47:...|abfss://c9364e95-...|
| B50005|  R700005|2026-06-08|Rs. 108,803.71| CORPORATE|2026-10-13 10:47:...|abfss://c9364e95-...|
| B50543|  R700551|2026-06-23|     15,328.74| Corporate|2026-10-13 10:47:...|abfss://c9364e95-...|
| B50656|  R700675|2026-06-30|     14,651.60| CORPORATE|2026-10-13 10:47:...|abfss://c9364e95-...|
| B50459|  R700463|2026-06-19| Rs. 10,086.61|  CONTRACT|2026-10-13 10:47:...|abfss://c9364e95-...|
| B50775| 

In [27]:
# check & verify the duplicated data

silver_billing.groupBy('bill_id').count().filter(F.col('count')>=2).show()

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 36, Finished, Available, Finished, False)

+-------+-----+
|bill_id|count|
+-------+-----+
+-------+-----+



In [28]:
silver_billing.describe().show()

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 37, Finished, Available, Finished, False)

+-------+-------+---------+----------+--------------+----------+--------------------+
|summary|BILL_ID|RENTAL_ID| BILL_DATE|    AMOUNT_INR|PAYER_TYPE|         source_file|
+-------+-------+---------+----------+--------------+----------+--------------------+
|  count|    787|      787|       787|           787|       787|                 787|
|   mean|   NULL|     NULL|      NULL|          NULL|      NULL|                NULL|
| stddev|   NULL|     NULL|      NULL|          NULL|      NULL|                NULL|
|    min| B50001|  R700001|2026-06-02|     10,774.66|  CONTRACT|abfss://c9364e95-...|
|    max| B50787|  R997692|2026-07-03|Rs. 991,647.76|   prepaid|abfss://c9364e95-...|
+-------+-------+---------+----------+--------------+----------+--------------------+



In [29]:
# cleaned amount_int & payer_type column

silver_billing = silver_billing.withColumn(
    'AMOUNT_INR',
    F.trim(F.regexp_replace('AMOUNT_INR','[^0-9]','')).cast('int')
)

silver_billing = silver_billing.withColumn(
    'PAYER_TYPE',
    F.upper('PAYER_TYPE'))

silver_billing.show(10)


StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 38, Finished, Available, Finished, False)

+-------+---------+----------+----------+----------+--------------------+--------------------+
|BILL_ID|RENTAL_ID| BILL_DATE|AMOUNT_INR|PAYER_TYPE|         ingested_at|         source_file|
+-------+---------+----------+----------+----------+--------------------+--------------------+
| B50612|  R700624|2026-06-23|   6415716|    DIRECT|2026-10-13 10:47:...|abfss://c9364e95-...|
| B50483|  R700487|2026-06-20|   2603789| CORPORATE|2026-10-13 10:47:...|abfss://c9364e95-...|
| B50362|  R700365|2026-06-22|  17489887|  CONTRACT|2026-10-13 10:47:...|abfss://c9364e95-...|
| B50005|  R700005|2026-06-08|  10880371| CORPORATE|2026-10-13 10:47:...|abfss://c9364e95-...|
| B50543|  R700551|2026-06-23|   1532874| CORPORATE|2026-10-13 10:47:...|abfss://c9364e95-...|
| B50656|  R700675|2026-06-30|   1465160| CORPORATE|2026-10-13 10:47:...|abfss://c9364e95-...|
| B50459|  R700463|2026-06-19|   1008661|  CONTRACT|2026-10-13 10:47:...|abfss://c9364e95-...|
| B50775|  R700892|2026-07-02|   1155549|    DIREC

In [30]:
silver_billing.printSchema()

print(f'{silver_billing.count()}','<< silver_billing_count')

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 39, Finished, Available, Finished, False)

root
 |-- BILL_ID: string (nullable = true)
 |-- RENTAL_ID: string (nullable = true)
 |-- BILL_DATE: string (nullable = true)
 |-- AMOUNT_INR: integer (nullable = true)
 |-- PAYER_TYPE: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

787 << silver_billing_count


In [31]:
# save billing data into delta table

silver_billing.write.\
format('delta')\
.mode('overwrite')\
.option('mergeSchema','true')\
.saveAsTable('silver.slv_billing')

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 40, Finished, Available, Finished, False)

### cleaned & Transformed depot table

In [32]:
silver_depot = spark.table('bronze.brz_depot')
silver_depot.show()

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 41, Finished, Available, Finished, False)

+----------+--------------+-----+----------+--------------------+--------------------+
|DEPOT_CODE|    DEPOT_NAME| ZONE|FLEET_SIZE|         ingested_at|         source_file|
+----------+--------------+-----+----------+--------------------+--------------------+
|      HADP|Hadapsar Depot| East|        55|2026-10-13 10:47:...|abfss://c9364e95-...|
|      CHKN|  Chakan Depot|North|        48|2026-10-13 10:47:...|abfss://c9364e95-...|
|      WAKD|   Wakad Depot| West|        42|2026-10-13 10:47:...|abfss://c9364e95-...|
|      KHRD| Kharadi Depot| East|        30|2026-10-13 10:47:...|abfss://c9364e95-...|
|      TLGN|Talegaon Depot|North|        25|2026-10-13 10:47:...|abfss://c9364e95-...|
|      BRMT|Baramati Depot|South|        20|2026-10-13 10:47:...|abfss://c9364e95-...|
+----------+--------------+-----+----------+--------------------+--------------------+



In [33]:
silver_depot.printSchema()

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 42, Finished, Available, Finished, False)

root
 |-- DEPOT_CODE: string (nullable = true)
 |-- DEPOT_NAME: string (nullable = true)
 |-- ZONE: string (nullable = true)
 |-- FLEET_SIZE: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [34]:
silver_depot = silver_depot.withColumn(
    'FLEET_SIZE',
    F.col('FLEET_SIZE').cast('int')
)

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 43, Finished, Available, Finished, False)

In [35]:
silver_depot.printSchema()

print(f'{silver_depot.count()}','<< silver_depot_count')

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 44, Finished, Available, Finished, False)

root
 |-- DEPOT_CODE: string (nullable = true)
 |-- DEPOT_NAME: string (nullable = true)
 |-- ZONE: string (nullable = true)
 |-- FLEET_SIZE: integer (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)

6 << silver_depot_count


In [36]:
# save data into delta  table

silver_depot.write\
.format('delta')\
.mode('overwrite')\
.option('mergeSchema','true')\
.saveAsTable('silver.slv_depot')

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 45, Finished, Available, Finished, False)

### cleaned & transfomed rental table

In [37]:
silver_rental = spark.table('bronze.brz_rental')
silver_rental.show(10)

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 46, Finished, Available, Finished, False)

+---------+-----------+----------+--------+-------------------+-------------------+-----------+--------------------+--------------------+
|rental_id|customer_id|depot_code|asset_id|        checkout_ts|         checkin_ts|rental_type|         ingested_at|         source_file|
+---------+-----------+----------+--------+-------------------+-------------------+-----------+--------------------+--------------------+
|  R700072|     C20213|      HADP|  GEN-50|2026-06-03 10:00:12|2026-06-05 00:24:12|   Priority|2026-10-13 10:47:...|abfss://c9364e95-...|
|  R700073|     C20044|      WAKD|  LDR-11|2026-06-03 06:30:18|2026-06-05 06:44:42|   Standard|2026-10-13 10:47:...|abfss://c9364e95-...|
|  R700074|     C20053|      HADP|  GEN-48|2026-06-03 20:15:10|2026-06-11 06:34:22|   STANDARD|2026-10-13 10:47:...|abfss://c9364e95-...|
|  R700075|     C20593|      KHRD|  EXC-19|2026-06-03 11:45:12|2026-06-12 23:30:48|   STANDARD|2026-10-13 10:47:...|abfss://c9364e95-...|
|  R700076|     C20242|      CHKN|

In [38]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

window_spec = Window.partitionBy('rental_id').orderBy(
    F.col('ingested_at').desc()
)

silver_rental = silver_rental.withColumn(
    'duplicate_flag',
    F.when(F.row_number().over(window_spec) == 1, 'KEEP')
     .otherwise('DUPLICATE')
)

silver_rental = silver_rental.withColumn(
    "checkin_ts",
    F.to_timestamp("checkin_ts", "yyyy-MM-dd HH:mm:ss")
)

silver_rental = silver_rental.withColumn(
    "checkout_ts",
    F.to_timestamp("checkout_ts", "yyyy-MM-dd HH:mm:ss")
)

display(silver_rental)

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 26126684-86a3-4db9-bde8-12229b8541bd)

In [40]:
silver_rental.printSchema()

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 49, Finished, Available, Finished, False)

root
 |-- rental_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- depot_code: string (nullable = true)
 |-- asset_id: string (nullable = true)
 |-- checkout_ts: timestamp (nullable = true)
 |-- checkin_ts: timestamp (nullable = true)
 |-- rental_type: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- duplicate_flag: string (nullable = false)



In [41]:
duplicate = silver_rental.filter(F.col('duplicate_flag')=='DUPLICATE')
display(duplicate)
duplicate.count()

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 50, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 29569031-f028-4b3b-a13f-3ef82f3385b1)

31

In [42]:
# Remove duplicates records & keep latest one

silver_rental = silver_rental.filter(F.col('duplicate_flag')=='KEEP')

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 51, Finished, Available, Finished, False)

In [43]:
silver_rental.groupBy('rental_id').count().filter(F.col('count')>=2).show()

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 52, Finished, Available, Finished, False)

+---------+-----+
|rental_id|count|
+---------+-----+
+---------+-----+



In [44]:
silver_rental = silver_rental.withColumn(
    'rental_type',
    F.initcap('rental_type')
)

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 53, Finished, Available, Finished, False)

In [45]:
from pyspark.sql import functions as F

# Identify impossible rentals where check-in is later than check-out
bad = F.col('checkin_ts') < F.col('checkout_ts')

# Count rentals with missing check-in before filtering
careless_dropped = silver_rental.filter(
    F.col('checkin_ts').isNull()
).count()

# Print how many still-out rentals exist
print("Still-out rentals:", careless_dropped)

# Move impossible records into a separate quarantine DataFrame; treat NULL conditions as False
quarantine_rental = silver_rental.filter(
    F.coalesce(bad, F.lit(False))
)

# Keep valid rentals and NULL check-ins, while removing impossible records
silver_rental = silver_rental.filter(
    ~F.coalesce(bad, F.lit(False))
)

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 54, Finished, Available, Finished, False)

Still-out rentals: 175


In [47]:
# double check there are not ant bad record & still null check-in cell keep after removing bad data

silver_rental.filter(
    F.col('checkin_ts') < F.col('checkout_ts')
).count()

silver_rental.filter(
    F.col('checkin_ts').isNull()
).count()



StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 57, Finished, Available, Finished, False)

175

In [48]:
silver_rental.show(10)

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 60, Finished, Available, Finished, False)

+---------+-----------+----------+--------+-------------------+-------------------+-----------+--------------------+--------------------+--------------+
|rental_id|customer_id|depot_code|asset_id|        checkout_ts|         checkin_ts|rental_type|         ingested_at|         source_file|duplicate_flag|
+---------+-----------+----------+--------+-------------------+-------------------+-----------+--------------------+--------------------+--------------+
|  R700001|     C20139|      TLGN|  CRN-21|2026-06-01 13:15:40|2026-06-06 00:18:04|   Priority|2026-10-13 10:47:...|abfss://c9364e95-...|          KEEP|
|  R700002|     C20057|      KHRD|  EXC-14|2026-06-01 19:30:36|2026-06-08 09:25:48|   Standard|2026-10-13 10:47:...|abfss://c9364e95-...|          KEEP|
|  R700003|     C20362|      WAKD|  EXC-22|2026-06-01 07:15:17|2026-06-08 00:03:17|   Standard|2026-10-13 10:47:...|abfss://c9364e95-...|          KEEP|
|  R700004|     C20398|      TLGN|  GEN-19|2026-06-01 08:00:55|2026-06-07 20:58:31

In [49]:
silver_rental.write\
.format('delta')\
.mode('overwrite')\
.option('overwriteSchema','true')\
.saveAsTable('silver.slv_rental')


StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 61, Finished, Available, Finished, True)

### join billing to rental with left join

In [50]:
slv_billing = spark.table('silver.slv_billing')
slv_rental = spark.table('silver.slv_rental')

# Keep every billing record and attach matching rental information
billing_joined = slv_billing.join(
    slv_rental,
    on='rental_id',
    how='left'
)



# Store bills that have no matching rental in a separate side table
unmatched_bills = billing_joined.filter(
    F.col('rental_id').isNull()
)



# Keep bills that have a matching rental
matched_bills = billing_joined.filter(
    F.col('rental_id').isNotNull()
)



StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 63, Finished, Available, Finished, True)

In [51]:
display(silver_rental)

StatementMeta(, 40fb9767-5a01-4aa8-9f43-0cab8561fb80, 64, Finished, Available, Finished, True)

SynapseWidget(Synapse.DataFrame, cc5c27ac-9f3b-4ae5-8c1a-0eb3a00a2463)